# Rerank Demo

This notebook shows two reranking approaches:
- LLM-based reranking
- Cross-Encoder reranking

We will first do a basic vector retrieval, then rerank the results to improve relevance.

In [ ]:
# Install required packages
!pip install faiss-cpu langchain langchain-community langchain-openai sentence-transformers python-dotenv

In [ ]:
import sys
from pathlib import Path

# Setup paths so we can import utils
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.env_loader import load_environment
load_environment()

import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.docstore.document import Document
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.retrievers import BaseRetriever
from typing import List, Any
from sentence_transformers import CrossEncoder

## Sample Data

In [ ]:
chunks = [
    "The capital of France is great.",
    "The capital of France is huge.",
    "The capital of France is beautiful.",
    """Have you ever visited Paris? It is a beautiful city where you can eat delicious food and see the Eiffel Tower. 
    I really enjoyed all the cities in france, but its capital with the Eiffel Tower is my favorite city.""", 
    "I really enjoyed my trip to Paris, France. The city is beautiful and the food is delicious. I would love to visit again. Such a great capital city."
]
docs = [Document(page_content=sentence) for sentence in chunks]
docs

## Create Vector Store

In [ ]:
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)

## Method 1: LLM-based Reranking

In [ ]:
class RatingScore(BaseModel):
    relevance_score: float = Field(..., description="The relevance score of a document to a query.")

def rerank_documents_llm(query: str, docs: List[Document], top_n: int = 3, model: str = "gpt-4o") -> List[Document]:
    prompt_template = PromptTemplate(
        input_variables=["query", "doc"],
        template="""On a scale of 1-10, rate the relevance of the following document to the query.
        Query: {query}
        Document: {doc}
        Relevance Score:"""
    )
    llm = ChatOpenAI(temperature=0, model_name=model)
    llm_chain = prompt_template | llm.with_structured_output(RatingScore)

    scored_docs = []
    for doc in docs:
        score = llm_chain.invoke({"query": query, "doc": doc.page_content}).relevance_score
        try:
            score = float(score)
        except ValueError:
            score = 0.0
        scored_docs.append((doc, score))

    reranked_docs = sorted(scored_docs, key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in reranked_docs[:top_n]]

In [ ]:
query = "what is the capital of france?"
initial_docs = vectorstore.similarity_search(query, k=5)
reranked_docs = rerank_documents_llm(query, initial_docs, top_n=3)

print("Top initial documents:")
for i, doc in enumerate(initial_docs[:3]):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)

print(f"\nQuery: {query}\n")
print("Top reranked documents:")
for i, doc in enumerate(reranked_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)

## Method 2: Cross-Encoder Reranking

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

class CrossEncoderRetriever(BaseRetriever, BaseModel):
    vectorstore: Any = Field(description="Vector store for initial retrieval")
    cross_encoder: Any = Field(description="Cross-encoder model for reranking")
    k: int = Field(default=5, description="Number of documents to retrieve initially")
    rerank_top_k: int = Field(default=3, description="Number of documents to return after reranking")

    class Config:
        arbitrary_types_allowed = True

    def get_relevant_documents(self, query: str) -> List[Document]:
        initial_docs = self.vectorstore.similarity_search(query, k=self.k)
        pairs = [[query, doc.page_content] for doc in initial_docs]
        scores = self.cross_encoder.predict(pairs)
        scored_docs = sorted(zip(initial_docs, scores), key=lambda x: x[1], reverse=True)
        return [doc for doc, _ in scored_docs[:self.rerank_top_k]]

    async def aget_relevant_documents(self, query: str) -> List[Document]:
        raise NotImplementedError("Async retrieval not implemented")

In [ ]:
cross_encoder_retriever = CrossEncoderRetriever(
    vectorstore=vectorstore,
    cross_encoder=cross_encoder,
    k=10,
    rerank_top_k=5
)

ce_docs = cross_encoder_retriever.get_relevant_documents(query)
print("Cross-encoder reranked documents:")
for i, doc in enumerate(ce_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)

## Why reranking helps

In [ ]:
print("Baseline vector search:")
for i, doc in enumerate(vectorstore.similarity_search(query, k=3)):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)

print("\n\nAfter LLM reranking:")
for i, doc in enumerate(reranked_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)

print("\n\nAfter Cross-Encoder reranking:")
for i, doc in enumerate(ce_docs):
    print(f"\nDocument {i+1}:")
    print(doc.page_content)